In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
data = data.drop(columns='Unnamed: 0')
print(len(data))

label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                    'front_running', 'reentrancy', 'time_manipulation',
                    'unchecked_low_calls']

selected_labels = label_columns[:8]

for column in label_columns:
    data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

# Giữ đúng 39645 rows
#data = data.iloc[:39645]
total_rows = len(data)
split_point = int(0.8 * total_rows)

print(split_point)
print(total_rows - split_point)

df_train = data.iloc[:split_point]
df_test = data.iloc[split_point:]


X_train_opcode = np.array(df_train.iloc[:, 11:].values)
X_test_opcode = np.array(df_test.iloc[:, 11:].values)

y_train = df_train[selected_labels].values
y_test = df_test[selected_labels].values
labels = data[selected_labels].values

45597
36477
9120


In [3]:
import numpy as np

print("Max:", np.max(X_train_opcode))
print("Min:", np.min(X_train_opcode))
print("Mean:", np.mean(X_train_opcode))

Max: 1520
Min: 0
Mean: 11.872067134280309


In [4]:
import torch
from torch.utils.data import Dataset

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [5]:
train_dataset = OpcodeDataset(X_train_opcode, y_train)
test_dataset = OpcodeDataset(X_test_opcode, y_test)

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [7]:
# import torch
# import torch.nn as nn

# class BiLSTMModel(nn.Module):
#     def __init__(self, vocab_size=2000, embedding_dim=384, hidden_dim=128, output_dim=8):
#         super(BiLSTMModel, self).__init__()
        
#         # Embedding
#         self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
#         # BiLSTM
#         self.lstm = nn.LSTM(
#             embedding_dim,
#             hidden_dim,
#             batch_first=True,
#             bidirectional=True
#         )
        
#         # Regularization
#         self.dropout = nn.Dropout(0.3)
#         self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
#         # Dense layers
#         self.fc1 = nn.Linear(hidden_dim * 2, 128)
#         self.bn2 = nn.BatchNorm1d(128)
#         self.fc2 = nn.Linear(128, 128)
        
#         # Output
#         self.out = nn.Linear(128, output_dim)
#         self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#         x = self.embedding(x)                 # (B, 280, 286)
        
#         _, (h_n, _) = self.lstm(x)
        
#         # Lấy hidden state của cả 2 chiều
#         x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
#         x = self.dropout(x)
#         x = self.bn1(x)
#         x = self.dropout(x)
        
#         x = self.fc1(x)
#         x = torch.relu(x)
#         x = self.bn2(x)
#         x = self.dropout(x)
        
#         x = self.fc2(x)
#         x = torch.relu(x)
        
#         x = self.out(x)
#         x = self.sigmoid(x)
        
#         return x

In [8]:
import torch
import torch.nn as nn

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=34000,
        embedding_dim=256,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        
        x = self.out(x)
        x = self.sigmoid(x)
        
        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMModel().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [10]:
def train(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).float()

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device).float()

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item()

    return total_loss / len(dataloader)

import torch

def get_predictions(model, dataloader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)  # đã có sigmoid
            
            all_preds.append(outputs.cpu())
            all_labels.append(y.cpu())

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return all_preds.numpy(), all_labels.numpy()

In [11]:
epochs = 50

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

    if (epoch + 1) % 10 == 0:
        y_pred_prob, y_true = get_predictions(model, test_loader, device)

        # threshold = 0.5
        y_pred = (y_pred_prob >= 0.5).astype(int)
        from sklearn.metrics import classification_report

        label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                    'front_running', 'reentrancy', 'time_manipulation',
                    'unchecked_low_calls']

        print(classification_report(y_true, y_pred, target_names=label_names))

Epoch 1: Train Loss=0.4273, Val Loss=0.3832
Epoch 2: Train Loss=0.3784, Val Loss=0.3473
Epoch 3: Train Loss=0.3550, Val Loss=0.3242
Epoch 4: Train Loss=0.3381, Val Loss=0.3213
Epoch 5: Train Loss=0.3240, Val Loss=0.3031
Epoch 6: Train Loss=0.3106, Val Loss=0.2977
Epoch 7: Train Loss=0.2999, Val Loss=0.2791
Epoch 8: Train Loss=0.2883, Val Loss=0.2731
Epoch 9: Train Loss=0.2795, Val Loss=0.2591
Epoch 10: Train Loss=0.2693, Val Loss=0.2555


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other       0.84      0.51      0.64      6091
     access_control       0.83      0.17      0.28       476
         arithmetic       0.92      0.93      0.92      5755
     denial_service       0.78      0.66      0.72      1268
      front_running       0.76      0.47      0.58      1018
         reentrancy       0.86      0.65      0.74      3798
  time_manipulation       0.76      0.34      0.47       458
unchecked_low_calls       0.91      0.87      0.89      4039

          micro avg       0.88      0.70      0.78     22903
          macro avg       0.83      0.57      0.65     22903
       weighted avg       0.87      0.70      0.76     22903
        samples avg       0.81      0.68      0.71     22903

Epoch 11: Train Loss=0.2627, Val Loss=0.2528
Epoch 12: Train Loss=0.2552, Val Loss=0.2460
Epoch 13: Train Loss=0.2479, Val Loss=0.2250
Epoch 14: Train Loss=0.2418, Val Loss=0.2309
Epoch 15: Train Loss=0.2

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other       0.88      0.90      0.89      6091
     access_control       0.63      0.27      0.38       476
         arithmetic       0.91      0.96      0.93      5755
     denial_service       0.80      0.70      0.75      1268
      front_running       0.69      0.60      0.64      1018
         reentrancy       0.89      0.83      0.86      3798
  time_manipulation       0.71      0.48      0.57       458
unchecked_low_calls       0.90      0.90      0.90      4039

          micro avg       0.88      0.86      0.87     22903
          macro avg       0.80      0.70      0.74     22903
       weighted avg       0.87      0.86      0.86     22903
        samples avg       0.82      0.82      0.80     22903

Epoch 21: Train Loss=0.2058, Val Loss=0.2211
Epoch 22: Train Loss=0.2025, Val Loss=0.2210
Epoch 23: Train Loss=0.1987, Val Loss=0.2204
Epoch 24: Train Loss=0.1956, Val Loss=0.2183
Epoch 25: Train Loss=0.1

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other       0.90      0.88      0.89      6091
     access_control       0.62      0.31      0.41       476
         arithmetic       0.93      0.94      0.93      5755
     denial_service       0.78      0.72      0.75      1268
      front_running       0.72      0.58      0.64      1018
         reentrancy       0.88      0.85      0.86      3798
  time_manipulation       0.74      0.46      0.57       458
unchecked_low_calls       0.91      0.88      0.89      4039

          micro avg       0.89      0.85      0.87     22903
          macro avg       0.81      0.70      0.74     22903
       weighted avg       0.88      0.85      0.86     22903
        samples avg       0.82      0.81      0.80     22903

Epoch 31: Train Loss=0.1736, Val Loss=0.2314
Epoch 32: Train Loss=0.1703, Val Loss=0.2254
Epoch 33: Train Loss=0.1691, Val Loss=0.2315
Epoch 34: Train Loss=0.1671, Val Loss=0.2294
Epoch 35: Train Loss=0.1

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other       0.89      0.88      0.89      6091
     access_control       0.57      0.34      0.43       476
         arithmetic       0.92      0.94      0.93      5755
     denial_service       0.76      0.77      0.77      1268
      front_running       0.67      0.62      0.64      1018
         reentrancy       0.88      0.85      0.86      3798
  time_manipulation       0.67      0.54      0.59       458
unchecked_low_calls       0.89      0.90      0.90      4039

          micro avg       0.87      0.86      0.87     22903
          macro avg       0.78      0.73      0.75     22903
       weighted avg       0.87      0.86      0.86     22903
        samples avg       0.83      0.82      0.81     22903

Epoch 41: Train Loss=0.1527, Val Loss=0.2375
Epoch 42: Train Loss=0.1517, Val Loss=0.2376
Epoch 43: Train Loss=0.1505, Val Loss=0.2430
Epoch 44: Train Loss=0.1479, Val Loss=0.2405
Epoch 45: Train Loss=0.1

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
torch.save(model.state_dict(), "DATASET/bilstm_model.pth")